# unbroadcast-pattern — worked example 1: Unbroadcast leading axes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbroadcast-pattern`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Forward broadcasting can prepend new leading axes to a tensor. The backward pass must undo that by summing those extra leading axes away. The rule: while `grad.ndim > original.ndim`, do `grad = grad.sum(dim=0)`, peeling one prepended axis per iteration until the rank matches.

## Worked solution

We restore a gradient to its pre-broadcast shape when only leading axes were added.

1. We compare ranks: `grad` has more dims than `original`, and broadcasting always adds new axes on the LEFT, so those extras are the leading ones.
2. `grad.sum(dim=0)` collapses the outermost extra axis; we loop until `grad.ndim == original.ndim`.
3. Summing (not averaging) is correct because the forward op replicated `original` across each leading position, so the gradient contributions add up.

With `original` shape `(3, 4)` and `grad` shape `(2, 3, 4)` of ones, the result is shape `(3, 4)` with every entry equal to 2 — the count of replicas summed.

In [ ]:
import torch as t

def unbroadcast_leading(grad: t.Tensor, original: t.Tensor) -> t.Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    return grad

original = t.zeros(3, 4)
grad = t.ones(2, 3, 4)
out = unbroadcast_leading(grad, original)
print('shape:', tuple(out.shape))
print('entry value:', out[0, 0].item())